<p>
<font size='5' face='Georgia, Arial'>IIC-2233 Apunte Programación Avanzada</font><br>
<font size='1'>&copy; 2015 Karim Pichara - Christian Pieringer. Todos los derechos reservados. Editado por Equipo Docente IIC2233 2018-1 al 2026-2</font>
</p>

# Tabla de contenidos

Este *notebook* es ***bonus***: no es parte de los contenidos de la semana, pero te puede servir para profundizar o para tu tarea.

1. [Ejecución de código de ejemplo del material que usa PyQt](#Ejecución-de-código-de-ejemplo-del-material-que-usa-PyQt)
2. [El patrón *worker*](#El-patrón-worker)
3. [Sonidos en PyQt6](#Sonidos-en-PyQt6)

## Ejecución de código de ejemplo del material que usa PyQt

Jupyter no fue creado para ejecutar interfaces gráficas de escritorio, por lo que en este _notebook_ el código se muestra como texto y **no** se ejecuta. Cada ejemplo está en la carpeta `scripts/`, y te recomendamos **fuertemente** ejecutarlo desde tu computador e interactuar con él.

**Importante:** al ejecutar un _script_, la terminal debe estar posicionada en la misma carpeta del archivo `.py`. Por ejemplo:

```bash
cd scripts/bonus-worker-y-sonidos
python3 1_move_to_thread.py
```

## El patrón *worker*

En el [*notebook* 3](3-qthread.ipynb) usamos el patrón más directo: heredar de `QThread` y escribir el trabajo en `run`. La documentación de Qt recomienda además otro patrón, que evita la confusión de afinidad que vimos en ese *notebook*: crear un `QObject` común (el *worker*) y **moverlo** a un `QThread` con `moveToThread`. Así, todos los métodos del *worker* conectados a señales se ejecutan en el *thread* secundario.

El ejemplo usa la misma función `donde_estoy()` del ejemplo de afinidad, que retorna si el código se está ejecutando en el *thread* principal o en uno secundario.

Este código se encuentra en el archivo `bonus-worker-y-sonidos/1_move_to_thread.py`.

```python
class Trabajador(QObject):
    """
    Un QObject común y corriente (NO hereda de QThread). Sus métodos se
    ejecutarán en el thread al que lo movamos con moveToThread.
    """

    senal_progreso = pyqtSignal(str)
    senal_terminado = pyqtSignal()

    def trabajar(self) -> None:
        print(f"[trabajar] me ejecuto en el {donde_estoy()}")
        for i in range(1, 6):
            sleep(0.5)
            self.senal_progreso.emit(f"Paso {i} de 5")
        self.senal_progreso.emit("Status: trabajo terminado")
        self.senal_terminado.emit()


class MiVentana(QWidget):
    def __init__(self) -> None:
        super().__init__()
        self.label = QLabel("Status: esperando", self)
        self.boton = QPushButton("Trabajar", self)
        self.label.setGeometry(10, 10, 230, 30)
        self.boton.setGeometry(10, 50, 230, 30)
        self.boton.clicked.connect(self.iniciar)
        self.setWindowTitle("Patrón worker + moveToThread")
        self.setGeometry(50, 50, 250, 90)

        # 1. Creamos un QThread "vacío": su run por defecto solo corre un
        #    event loop para ese thread
        self.mi_thread = QThread()
        # 2. Creamos el trabajador y lo movemos a ese thread
        self.trabajador = Trabajador()
        self.trabajador.moveToThread(self.mi_thread)
        # 3. Conectamos señales
        self.trabajador.senal_progreso.connect(self.label.setText)
        self.trabajador.senal_terminado.connect(self.mi_thread.quit)
        self.mi_thread.started.connect(self.trabajador.trabajar)
        self.show()

    def iniciar(self) -> None:
        if not self.mi_thread.isRunning():
            self.mi_thread.start()
```

Fíjate en que el `QThread` no se hereda: su `run` por defecto ejecuta un *event loop* para ese *thread*, y es ese *event loop* el que ejecuta `trabajar` cuando se emite `started`. Al terminar, el *worker* emite `senal_terminado`, que está conectada a `self.mi_thread.quit` para detener ese *event loop*.

## Sonidos en PyQt6

PyQt6 ofrece dos clases para reproducir audio, en el módulo `PyQt6.QtMultimedia`:

| | `QSoundEffect` | `QMediaPlayer` |
|---|---|---|
| Formatos | Solo `.wav` | Muchos, incluyendo `.mp3` |
| Configuración | `setSource(url)` y listo | Necesita además un `QAudioOutput` |
| Uso típico | Efectos cortos que deben sonar sin retraso | Música y audios largos o comprimidos |

En el siguiente ejemplo, el botón "Wooow" reproduce un efecto corto en MP3 (por eso usa `QMediaPlayer`), y los otros botones inician y detienen una música de fondo en WAV con `QSoundEffect`, repitiéndola indefinidamente con `setLoopCount`.

Ambas clases reciben la ubicación del archivo como un `QUrl`, que se crea con `QUrl.fromLocalFile(path)`. Para `QMediaPlayer` construimos un *path* absoluto con `os.path.abspath` (del módulo `os`), a partir del *path* relativo.

Este código se encuentra en el archivo `bonus-worker-y-sonidos/2_sonidos.py`.

```python
        # Opción MP3: QMediaPlayer (reproduce muchos formatos, como MP3)
        # 1. Para que QMediaPlayer reproduzca audio, se debe agregar un QAudioOutput
        # 2. Usamos el path absoluto. Para generarlo usamos os.path.abspath
        #    (del módulo os), que transforma el path relativo en absoluto
        # 3. Se utiliza setSource para indicar el path al audio a escuchar
        self.player = QMediaPlayer(self)
        self.audio_output = QAudioOutput(self)
        self.player.setAudioOutput(self.audio_output)
        path = os.path.abspath(join("sounds", "waku-waku.mp3"))
        self.player.setSource(QUrl.fromLocalFile(path))

        # Opción WAV: QSoundEffect (solo reproduce archivos WAV)
        self.media_player_wav = QSoundEffect(self)
        self.media_player_wav.setVolume(0.1)  # Opcional
        # Como es música de fondo, la repetimos indefinidamente
        self.media_player_wav.setLoopCount(QSoundEffect.Loop.Infinite.value)
        file_url = QUrl.fromLocalFile(join("sounds", "see-you-again.wav"))
        self.media_player_wav.setSource(file_url)

        # Mostrar ventana
        self.show()

    def empezar_sonido_sorpresa(self) -> None:
        self.player.play()

    def empezar_musica_fondo(self) -> None:
        if not self.media_player_wav.isPlaying():
            self.media_player_wav.play()

    def parar_musica_fondo(self) -> None:
        if self.media_player_wav.isPlaying():
            self.media_player_wav.stop()
```

Algunas notas:

- Al reproducir un MP3 puede aparecer en la consola un *warning* como *Could not update timestamps for skipped samples*. Se debe a cómo está comprimido el formato, y no afecta en nada.
- En Windows, si el *path* absoluto supera los 260 caracteres, el sonido no se reproduce. Evita guardar tus códigos en carpetas muy profundas.
- Si buscas ejemplos en internet, mucho código usa PyQt5, donde `QMediaPlayer` se configuraba con `QMediaContent` y `setMedia`. En PyQt6 se usa `QAudioOutput` y `setSource`, como en este ejemplo.